# Chapter 4 — Semantic Search from Scratch

This notebook accompanies **Chapter 4** of *Build an Advanced RAG Application (From Scratch)*.

We start with a corpus of hotel reviews and build a semantic search engine in three increasingly fast forms:

1. **Pure NumPy** — cosine similarity by hand.
2. **NumPy with normalized Euclidean distance** — same ranking, different metric.
3. **FAISS** — the production-grade vector search library.

We finish by comparing FAISS index types (Flat / HNSW / IVF-PQ) on the same query.

> Reusable code lives in `data_loader.py` and `search.py` next to this notebook.


## 0. Setup

Make sure you're in the `advanced-rag` conda env (see the root README) and that the kernel for this notebook points to it.

In [1]:
import os, sys, time

# macOS: faiss-cpu and torch each bundle their own libomp.dylib; loading both
# in one process causes a silent kernel segfault during FAISS kmeans training
# (e.g. IVF-PQ). KMP_DUPLICATE_LIB_OK silences the abort; OMP_NUM_THREADS=1
# avoids the runtime race. Both must be set before `import torch`.
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
import torch
import faiss
faiss.omp_set_num_threads(1)

# Make the chapter folder importable
sys.path.insert(0, os.path.dirname(os.path.abspath('.')) if not os.path.exists('search.py') else '.')

from data_loader import load_paris_reviews
from search import (
    load_embedding_model,
    get_embeddings,
    cosine_search,
    euclidean_search,
    build_faiss_cosine_index,
    search_faiss_index,
    build_faiss_indices,
)


## 1. Load the Paris hotel reviews

The dataset is hosted on the HuggingFace Hub. The first call downloads it; subsequent calls hit the local cache.

In [2]:
df_paris = load_paris_reviews()
print(f"Rows: {len(df_paris):,}")
df_paris.head()

Rows: 1,200


,hotel_name,hotel_description,review_title,review_text,rate,tripdate,hotel_url,hotel_image,price_range,rating_value,review_count,street_address,locality,country
0,Hotel Malte - Astotel,Located in the 2nd district next to the Stock ...,Awesome Paris Hotel!,"Fantastic hotel! Awesome location, great chara...",5.0,January 2024,https://www.tripadvisor.com/Hotel_Review-g1871...,https://media-cdn.tripadvisor.com/media/photo-...,$$ (Based on Average Nightly Rates for a Stand...,5.0,2985,63 rue de Richelieu,Paris,France
1,Hotel Malte - Astotel,Located in the 2nd district next to the Stock ...,Charming Hotel,Charming Hotel in a central location. The sta...,5.0,May 2023,https://www.tripadvisor.com/Hotel_Review-g1871...,https://media-cdn.tripadvisor.com/media/photo-...,$$ (Based on Average Nightly Rates for a Stand...,5.0,2985,63 rue de Richelieu,Paris,France
2,Hotel Malte - Astotel,Located in the 2nd district next to the Stock ...,Highly recommend this hotel,Highly recommend this hotel and we would absol...,5.0,December 2023,https://www.tripadvisor.com/Hotel_Review-g1871...,https://media-cdn.tripadvisor.com/media/photo-...,$$ (Based on Average Nightly Rates for a Stand...,5.0,2985,63 rue de Richelieu,Paris,France
3,Hotel Malte - Astotel,Located in the 2nd district next to the Stock ...,"Good location, excellent staff and large room",Good central location - close to Metro and man...,5.0,December 2023,https://www.tripadvisor.com/Hotel_Review-g1871...,https://media-cdn.tripadvisor.com/media/photo-...,$$ (Based on Average Nightly Rates for a Stand...,5.0,2985,63 rue de Richelieu,Paris,France
4,Hotel Malte - Astotel,Located in the 2nd district next to the Stock ...,"Good staff, quiet and beautiful location.","Lovely staff. All were good, and Manon was out...",5.0,January 2024,https://www.tripadvisor.com/Hotel_Review-g1871...,https://media-cdn.tripadvisor.com/media/photo-...,$$ (Based on Average Nightly Rates for a Stand...,5.0,2985,63 rue de Richelieu,Paris,France


In [3]:
df_paris.hotel_name.value_counts().head(10)

hotel_name
Hotel Malte - Astotel           40
Hotel Astoria - Astotel         40
Novotel Paris Les Halles        40
La Maison Favart                40
Grand Hotel du Palais Royal     40
Hotel Maison Mere               40
Hotel des Arts - Montmartre     40
Hotel Joke - Astotel            40
Passy Eiffel Hotel              40
Best Western Plus La Demeure    40
Name: count, dtype: int64

## 2. Embed the reviews

We use `nomic-ai/nomic-embed-text-v1.5` — a strong open-weight 768-dim embedding model. On CPU this takes a few minutes; on GPU it's much faster.

In [4]:
model = load_embedding_model()

if torch.cuda.is_available():
    model = model.to("cuda")
    print("CUDA available — model on GPU.")
elif torch.backends.mps.is_available():
    model = model.to("mps")
    print("MPS available — model on Apple Silicon GPU.")
else:
    print("Running on CPU.")

<All keys matched successfully>


MPS available — model on Apple Silicon GPU.


In [5]:
reviews = df_paris["review_text"].tolist()

# nomic was trained with task prefixes (see Chapter 3): every document carries
# "search_document: " and every query carries "search_query: ".
docs = ["search_document: " + r for r in reviews]
review_embeddings = model.encode(docs, show_progress_bar=True).astype("float32")
print(f"Embeddings shape: {review_embeddings.shape}")

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


Embeddings shape: (1200, 768)


## 3. Search by hand: cosine similarity

No libraries needed. We compute dot products of L2-normalized vectors.

In [6]:
query = "Hotel with a view of the Eiffel tower."
query_embedding = model.encode(["search_query: " + query]).astype("float32")

t0 = time.time()
indices, sims = cosine_search(query_embedding, review_embeddings, k=5)
print(f"Cosine search took {time.time()-t0:.4f}s")

print(f"\nQuery: {query}\n")
for rank, (idx, sim) in enumerate(zip(indices, sims), 1):
    print(f"{rank}. {df_paris.iloc[idx]['hotel_name']}  (sim={sim:.4f})")
    print(f"   {df_paris.iloc[idx]['review_text'][:200]}...\n")

Cosine search took 0.0027s

Query: Hotel with a view of the Eiffel tower.

1. Citadines Tour Eiffel Paris  (sim=0.8340)
   This property is an Affordable hotel apartment in Paris city center close to Eiffel Tower. We got the room with Eiffel Tower view, our view was the top half of the tower. It was a bit expensive but we...

2. Passy Eiffel Hotel  (sim=0.8332)
   The hotel is near Eiffel tower and we could see the tower from our hotel balcony. The staff are very friendly and helpful and the breakfast is outstanding. We would like to stay here again when we com...

3. Cler Hotel  (sim=0.8260)
   Very nice hotel! Staff is great, very accommodating. Would come back again. It is pretty small- the rooms are smaller than they look online. Eiffel tower room has a great view but the other rooms have...

4. Grand Hotel du Palais Royal  (sim=0.8246)
   Beautiful hotel with exceptional staff and service in a perfect location. We loved our balcony with views of the Eiffel Tower and the Basilica. 

## 4. Same ranking via normalized Euclidean distance

For unit-norm vectors, `||a − b||² = 2(1 − cos(a, b))`, so Euclidean distance ranks identically to cosine similarity (just inverted).

In [7]:
indices, distances = euclidean_search(query_embedding, review_embeddings, k=5)
for rank, (idx, dist) in enumerate(zip(indices, distances), 1):
    print(f"{rank}. {df_paris.iloc[idx]['hotel_name']}  (dist={dist:.4f})")

1. Citadines Tour Eiffel Paris  (dist=0.5762)
2. Passy Eiffel Hotel  (dist=0.5775)
3. Cler Hotel  (dist=0.5900)
4. Grand Hotel du Palais Royal  (dist=0.5923)
5. Hotel Marignan Champs-Elysees  (dist=0.5978)


## 5. Scale up with FAISS

NumPy is fine for a few thousand rows but quickly falls over. FAISS is purpose-built for vector search.

We build an `IndexFlatIP` (inner product) on **L2-normalized** vectors — that's mathematically equivalent to exact cosine similarity but with FAISS's optimized SIMD search.

In [8]:
faiss_index = build_faiss_cosine_index(review_embeddings)

t0 = time.time()
distances, indices = search_faiss_index(query_embedding, faiss_index, k=5)
print(f"FAISS search took {time.time()-t0:.4f}s\n")

for rank, (idx, sim) in enumerate(zip(indices[0], distances[0]), 1):
    print(f"{rank}. {df_paris.iloc[idx]['hotel_name']}  (cos={sim:.4f})")

FAISS search took 0.0015s

1. Citadines Tour Eiffel Paris  (cos=0.8340)
2. Passy Eiffel Hotel  (cos=0.8332)
3. Cler Hotel  (cos=0.8260)
4. Grand Hotel du Palais Royal  (cos=0.8246)
5. Hotel Marignan Champs-Elysees  (cos=0.8213)


### Aggregate by hotel

A single hotel may show up many times in the top-k. Group by hotel and rank by mean cosine to surface *places* rather than individual reviews.

In [9]:
k = 120
distances, indices = search_faiss_index(query_embedding, faiss_index, k=k)

hotels = {}
for idx, dist in zip(indices[0], distances[0]):
    name = df_paris.iloc[idx]["hotel_name"]
    h = hotels.setdefault(name, {"reviews": [], "scores": []})
    h["reviews"].append(df_paris.iloc[idx]["review_text"])
    h["scores"].append(float(dist))

ranked = sorted(
    [(n, np.mean(h["scores"]), len(h["reviews"])) for n, h in hotels.items() if len(h["reviews"]) >= 2],
    key=lambda t: t[1],
    reverse=True,
)
for name, mean_score, n in ranked[:10]:
    print(f"{mean_score:.4f}  {n:3d} reviews  {name}")

0.8032    5 reviews  Hotel Marignan Champs-Elysees
0.7891   16 reviews  Citadines Tour Eiffel Paris
0.7885   23 reviews  Hotel La Comtesse
0.7880   15 reviews  Passy Eiffel Hotel
0.7876   19 reviews  Pullman Paris Eiffel Tower Hotel
0.7861   12 reviews  Cler Hotel
0.7819    8 reviews  citizenM Paris Champs-Elysees
0.7799    2 reviews  Best Western Plus La Demeure
0.7795    3 reviews  Hotel du Danube Saint Germain
0.7784    9 reviews  Hotel Tourisme Avenue


### Rerank the candidates with a cross-encoder

The bi-encoder scores every review with one precomputed dot product, which is fast but blends everything a review is about into a single vector. A **cross-encoder** reads the query and a review *together* and scores how directly the review answers the query. It is far more accurate but cannot be precomputed, so we only run it on the bi-encoder's shortlist: retrieve 50 fast, then rerank those 50 well.

In [10]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# First stage: bi-encoder returns 50 candidates fast
distances, indices = search_faiss_index(query_embedding, faiss_index, k=50)
candidates = indices[0]

# Second stage: cross-encoder scores each (query, review) pair jointly.
# The cross-encoder reads raw text, so no nomic task prefix here.
pairs = [[query, df_paris.iloc[i]["review_text"]] for i in candidates]
rerank_scores = reranker.predict(pairs)
order = rerank_scores.argsort()[::-1]

print("Bi-encoder top 5 (by cosine):")
for rank, i in enumerate(candidates[:5], 1):
    print(f"{rank}. cos={distances[0][rank-1]:.4f}  {df_paris.iloc[i]['hotel_name']}")

print("\nCross-encoder top 5 (after rerank):")
for rank, j in enumerate(order[:5], 1):
    i = candidates[j]
    print(f"{rank}. ce={rerank_scores[j]:.2f}  {df_paris.iloc[i]['hotel_name']}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Bi-encoder top 5 (by cosine):
1. cos=0.8340  Citadines Tour Eiffel Paris
2. cos=0.8332  Passy Eiffel Hotel
3. cos=0.8260  Cler Hotel
4. cos=0.8246  Grand Hotel du Palais Royal
5. cos=0.8213  Hotel Marignan Champs-Elysees

Cross-encoder top 5 (after rerank):
1. ce=8.58  Hotel La Comtesse
2. ce=8.26  Hotel Marignan Champs-Elysees
3. ce=8.22  Passy Eiffel Hotel
4. ce=8.17  Pullman Paris Eiffel Tower Hotel
5. ce=8.00  Hotel La Comtesse


## 6. Comparing FAISS index types

- **Flat** — exact, full-scan; slow on big corpora.
- **HNSW** — graph-based, fast & accurate, more memory.
- **IVF-PQ** — clustered + quantized, very fast, lossy.

For a small corpus the latency differences are tiny; on millions of vectors they're decisive.

In [11]:
indices_set = build_faiss_indices(review_embeddings)
k = 100

for name, idx in indices_set.items():
    t0 = time.time()
    distances, top = idx.search(query_embedding, k)
    print(f"{name:6s}: {time.time()-t0:.4f}s  unique hotels = {len({df_paris.iloc[i]['hotel_name'] for i in top[0]})}")

WARNING clustering 1200 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 1200 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 1200 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 1200 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 1200 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 1200 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 1200 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 1200 points to 256 centroids: please provide at least 9984 training points


flat  : 0.0006s  unique hotels = 14
hnsw  : 0.0008s  unique hotels = 14
ivfpq : 0.0011s  unique hotels = 12


## 7. The retriever as one reusable function

The offline work (load, encode, index) runs once at startup. What every later chapter needs is a single entry point: a query string in, ranked hotels out. This is the search layer Chapter 5 puts a decoder behind.

In [12]:
def retrieve(query, k=5, depth=120):
    query_embedding = model.encode(["search_query: " + query]).astype("float32")
    distances, indices = search_faiss_index(query_embedding, faiss_index, k=depth)
    hotels = {}
    for i, score in zip(indices[0], distances[0]):
        name = df_paris.iloc[i]["hotel_name"]
        hotels.setdefault(name, []).append(float(score))
    ranked = sorted(
        [(name, float(np.mean(s)), len(s))
         for name, s in hotels.items() if len(s) >= 2],
        key=lambda t: t[1], reverse=True,
    )
    return ranked[:k]


for name, score, n in retrieve("a room with a view of the Eiffel Tower"):
    print(f"{score:.3f}  {n:3d} reviews  {name}")

0.751   17 reviews  Pullman Paris Eiffel Tower Hotel
0.746   23 reviews  Hotel La Comtesse
0.742   18 reviews  Citadines Tour Eiffel Paris
0.741   13 reviews  Passy Eiffel Hotel
0.740    6 reviews  Hotel Marignan Champs-Elysees


### Persist the index so the offline work runs once

As written, restarting re-encodes the corpus and rebuilds the index every time. Save both to disk once, then a restart loads a ready retriever in a moment instead of re-encoding for minutes.

In [ ]:
# Save once, after building
np.save("review_embeddings.npy", review_embeddings)
faiss.write_index(faiss_index, "hotels.faiss")

# On the next startup, load instead of rebuild
review_embeddings = np.load("review_embeddings.npy")
faiss_index = faiss.read_index("hotels.faiss")
print("saved and reloaded:", review_embeddings.shape, faiss_index.ntotal, "vectors")

## What's next

Chapter 5 plugs the **decoder** (LLM) on top of these retrievals to start producing grounded answers — the first half of a full RAG pipeline. Chapter 6 wires retrieval + generation together end-to-end and adds a real vector database (Qdrant).